# Практическая работа №1. Часть 4А
## Архитектуры сверточных сетей

**Цель:** собрать из сверток, ReLU и пулинга настоящую сверточную сеть в виде класса `nn.Module`, обучить ее на FashionMNIST и сравнить с полносвязной сетью по числу параметров и точности; посмотреть, что «видят» ее слои, и проверить на опыте, зачем нужны остаточные связи ResNet.

> **Правило части.** Теперь используем готовые модули PyTorch: `nn.Module`, `nn.Conv2d`, `nn.MaxPool2d`, `nn.Linear`, `DataLoader`, `torch.optim`. Свои функции из частей 1С и 3К (`count_params` для списка слоев, `out_size`, `conv_params`) служат для проверки – они повторены в первой ячейке.

**Задача:** классифицировать картинки одежды $28 \times 28$ из набора FashionMNIST на 10 классов. Сквозной пример со слайдов: полносвязная сеть 784 → 128 → 10 ($101\,770$ параметров) против сверточной сети `SimpleCNN` ($9\,098$ параметров).

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается»; для точности обученных сетей указан нижний порог. Ячейки обучения выполняются 1–3 минуты. Сначала прочитайте теорию ниже.

## Как устроена сверточная сеть

**1. Экстрактор признаков и классификатор.** Типовая сверточная сеть (CNN) состоит из двух частей:
- **экстрактор признаков** – несколько повторяющихся блоков «свертка → ReLU → пулинг». Каждый блок уменьшает картинку по высоте и ширине (пулинг) и увеличивает число каналов (свертка с большим числом фильтров): пространственная информация «переливается» в каналы-признаки;
- **классификатор** – карта признаков вытягивается в вектор (`flatten`) и подается в полносвязный слой, который выдает логиты классов, как в части 1С.

**2. Наша сеть `SimpleCNN` и формы тензоров.** По формуле размера выхода из части 3К $n_{out} = \lfloor (n + 2p - k)/s \rfloor + 1$:

| Слой | Что делает | Форма выхода для одной картинки |
|---|---|---|
| вход | черно-белая картинка | $1 \times 28 \times 28$ |
| `Conv2d(1, 8, 3, padding=1)` + ReLU | 8 фильтров $3 \times 3$, $(28 + 2 - 3)/1 + 1 = 28$ | $8 \times 28 \times 28$ |
| `MaxPool2d(2)` | максимум в окнах $2 \times 2$ | $8 \times 14 \times 14$ |
| `Conv2d(8, 16, 3, padding=1)` + ReLU | 16 фильтров $8 \times 3 \times 3$ | $16 \times 14 \times 14$ |
| `MaxPool2d(2)` | | $16 \times 7 \times 7$ |
| `flatten` | вектор | $16 \cdot 7 \cdot 7 = 784$ |
| `Linear(784, 10)` | логиты 10 классов | $10$ |

**3. Сколько параметров.** Сверточный слой – $C_{out}(C_{in} k^2) + C_{out}$ (часть 3К), полносвязный – $n_{in} n_{out} + n_{out}$ (часть 1С):

$$\underbrace{8 \cdot 9 + 8}_{80} + \underbrace{16 \cdot 72 + 16}_{1\,168} + \underbrace{784 \cdot 10 + 10}_{7\,850} = 9\,098.$$

У полносвязной сети 784 → 128 → 10 параметров $101\,770$ – в 11 раз больше. Зато вычислений у CNN больше: каждое ядро применяется во многих позициях. Умножений на одну картинку: у CNN $8 \cdot 28^2 \cdot 9 + 16 \cdot 14^2 \cdot 72 + 784 \cdot 10 = 290\,080$, у MLP $784 \cdot 128 + 128 \cdot 10 = 101\,632$.

*Почему маленькая CNN обгоняет большую MLP.* В CNN заложены знания о картинках (Гудфеллоу называет это «бесконечно сильным априорным распределением», раздел 9.4): соседние пиксели важны вместе (локальность), один узор ищется везде одним ядром (разделение параметров, раздел 7.9.1), небольшие сдвиги не важны (пулинг). MLP должна выучить все это из данных, тратя параметры на каждый пиксель.

**4. Что «видят» слои.** Число на карте признаков зависит от окна исходной картинки – **рецептивного поля**. После первой свертки $3 \times 3$ это $3 \times 3$ пикселя, после пулинга и второй свертки – уже $8 \times 8$, после второго пулинга – $10 \times 10$. Поэтому первые слои находят простые детали – границы и перепады яркости, – а следующие комбинируют их в углы, текстуры и части предметов (подошву, рукав, горловину).

**5. Эволюция архитектур** (Гудфеллоу, раздел 9.11):
- **LeNet** (1998) – распознавание рукописных цифр: свертки и пулинг, около 60 тыс. параметров – по устройству почти наша `SimpleCNN`;
- **AlexNet** (2012) – победа на ImageNet (1,2 млн картинок, 1000 классов): вдвое меньше ошибок, чем у лучших «ручных» методов. Новое: ReLU вместо сигмоиды, dropout против переобучения (Б2.3), обучение на видеокартах; около 60 млн параметров;
- **VGG** (2014) – только свертки $3 \times 3$, собранные в стопки. Две свертки $3 \times 3$ подряд видят окно $5 \times 5$, как одна $5 \times 5$, но весов у них $2 \cdot 9 = 18$ против 25 на пару каналов, и между ними есть еще одна ReLU;
- **ResNet** (2015) – остаточные связи, сети из 50–152 слоев, ошибка на ImageNet ниже человеческой оценки.

**6. Остаточная связь.** Эксперименты показали парадокс: обычная сеть из 56 слоев учится **хуже**, чем из 20, даже на обучающей выборке. Дело не в переобучении, а в том, что глубокую сеть трудно оптимизировать. Решение ResNet – блок, который вычисляет не новое значение, а **добавку** к входу:

$$y = F(x) + x,$$

где $F$ – две свертки с ReLU. Два следствия:
- блоку легко ничего не делать: достаточно $F(x) = 0$, то есть нулевых весов, – поэтому добавление блоков не может сделать сеть хуже;
- градиент проходит без потерь. По цепному правилу из части 2Г $\dfrac{\partial y}{\partial x} = \dfrac{\partial F}{\partial x} + 1$. В обычной сети множители $\dfrac{\partial F}{\partial x}$ перемножаются по всем слоям: если каждый слой ослабляет градиент в 10 раз, через 8 слоев останется $0{,}1^8 = 10^{-8}$. В остаточной сети каждый множитель $1 + 0{,}1$, и через 8 блоков градиент равен $1{,}1^8 \approx 2{,}14$ – он доходит до первых слоев. В настоящих ResNet внутри блока есть еще пакетная нормировка BatchNorm – о ней в Б2.2.

**7. Класс `nn.Module`.** Сеть в PyTorch – это класс-наследник `nn.Module` с двумя методами:
- `__init__` создает слои и сохраняет их **атрибутами** (`self.conv1 = nn.Conv2d(...)`); PyTorch сам находит их веса – они появляются в `model.parameters()` и попадают в оптимизатор;
- `forward(x)` описывает порядок вычислений. Вызов `model(x)` запускает `forward` (напрямую `forward` не вызывают).

Несколько слоев подряд удобно объединять в `nn.Sequential`. Методы `model.train()` и `model.eval()` переключают режим – это важно для dropout и BatchNorm (Блок 2).

**8. Данные: `Dataset` и `DataLoader`.** `datasets.FashionMNIST` хранит 60 000 обучающих и 10 000 тестовых картинок; преобразование `transforms.ToTensor()` превращает картинку в тензор $1 \times 28 \times 28$ с числами от 0 до 1. `DataLoader` нарезает набор на пакеты и перемешивает их каждую эпоху (`shuffle=True`) – то, что в части 2Г делала функция `train_epoch` через `torch.randperm`. Каждый шаг цикла `for x, y in loader` дает пакет картинок $(64, 1, 28, 28)$ и меток $(64,)$.

**9. Обучение.** Цикл обучения тот же, что в части 2Г: `zero_grad → forward → loss → backward → step`, потеря – `nn.CrossEntropyLoss` по логитам. Оптимизатор возьмем **Adam** (он сам подбирает шаг для каждого веса и учится быстрее обычного SGD; подробно – в Б2.1).

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
from pathlib import Path
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

# FashionMNIST заранее скачивается в папку data (интернет в аудитории не нужен):
#   datasets.FashionMNIST('data', train=True, download=True); datasets.FashionMNIST('data', train=False, download=True)
DATA_DIR = Path('data')
tf = transforms.ToTensor()
try:
    train_ds = datasets.FashionMNIST(DATA_DIR, train=True, download=False, transform=tf)
    test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=False, transform=tf)
    REAL_DATA = True
except Exception:
    print('FashionMNIST не найден в папке data – используются случайные картинки FakeData, точность будет около 0.1')
    train_ds = datasets.FakeData(size=6000, image_size=(1, 28, 28), num_classes=10, transform=tf)
    test_ds = datasets.FakeData(size=1000, image_size=(1, 28, 28), num_classes=10, transform=tf, random_offset=6000)
    REAL_DATA = False
print('обучение:', len(train_ds), ' тест:', len(test_ds))

g = torch.Generator().manual_seed(RANDOM_STATE)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=g)
train_eval_loader = DataLoader(train_ds, batch_size=256)      # для оценки точности на обучающей выборке
test_loader = DataLoader(test_ds, batch_size=256)

CLASSES = ['футболка', 'брюки', 'свитер', 'платье', 'пальто', 'сандалия', 'рубашка', 'кроссовок', 'сумка', 'ботинок']

# Функции из частей 1С и 3К – для проверки
def count_params_list(layer_sizes):
    return sum(a * b + b for a, b in zip(layer_sizes[:-1], layer_sizes[1:]))

def out_size(n, k, p=0, s=1):
    return (n + 2 * p - k) // s + 1

def conv_params(c_in, c_out, k):
    return c_out * (c_in * k * k) + c_out

In [ ]:
# Учебный пример со слайдов
x_batch, y_batch = next(iter(test_loader))
print(tuple(x_batch.shape), tuple(y_batch.shape), '  ожидается (256, 1, 28, 28) (256,)')
print(round(x_batch.min().item(), 2), round(x_batch.max().item(), 2), '  ожидается 0.0 1.0 (ToTensor делит на 255)')

# Полносвязная сеть из части 1С: 784 -> 128 -> 10
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.net(x)

# Формы по формуле из части 3К: 28 -> 28 -> 14 -> 14 -> 7
print(out_size(28, 3, p=1), out_size(28, 2, s=2), out_size(14, 3, p=1), out_size(14, 2, s=2), '  ожидается 28 14 14 7')

**Задание А1. Класс `SimpleCNN`.**

Напишите сеть из п. 2 теории как класс-наследник `nn.Module` (п. 7):
- в `__init__` после строки `super().__init__()` (она обязательна: запускает настройку базового класса) создайте два атрибута:
  - `self.features` – `nn.Sequential` из шести модулей: `nn.Conv2d(1, 8, 3, padding=1)`, `nn.ReLU()`, `nn.MaxPool2d(2)`, `nn.Conv2d(8, 16, 3, padding=1)`, `nn.ReLU()`, `nn.MaxPool2d(2)`;
  - `self.classifier` – `nn.Linear(16 * 7 * 7, 10)`;
- в `forward(x)`: пропустить `x` через `features`, вытянуть каждую картинку пакета в вектор методом `x.flatten(1)` (объединяет все оси, начиная с первой: $(N, 16, 7, 7) \to (N, 784)$; ось пакета 0 остается) и подать в `classifier`.

Порядок модулей в `features` важен: в задании А5 мы будем брать его начало срезом `model.features[:2]` – «первая свертка и ReLU».

*Типичная ошибка:* написать `x.flatten()` без аргумента – тогда весь пакет превратится в один длинный вектор, и `Linear` выдаст ошибку формы.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ...                                   # <-- свертка 1 -> 8, ReLU, пулинг, свертка 8 -> 16, ReLU, пулинг
        )
        self.classifier = ...                     # <-- полносвязный слой 16*7*7 -> 10

    def forward(self, x):
        x = ...                                   # <-- экстрактор признаков
        x = ...                                   # <-- вытянуть в вектор, начиная с оси 1
        return ...                                # <-- логиты классов


torch.manual_seed(RANDOM_STATE)
cnn = SimpleCNN()
print(cnn)
x0 = torch.zeros(4, 1, 28, 28)
print(tuple(cnn.features(x0).shape), tuple(cnn(x0).shape), '  ожидается (4, 16, 7, 7) (4, 10)')

**Вопрос А1.** а) Почему число каналов растет от слоя к слою (1 → 8 → 16), а размер карты падает (28 → 14 → 7)? б) Что сломается, если убрать `padding=1` в обеих свертках? Посчитайте новые размеры и новый вход классификатора.

*Ваш ответ:*

**Задание А2. Сколько параметров у модели.**

Функция `count_params(model)` возвращает число обучаемых параметров любой модели: `model.parameters()` перебирает все тензоры весов и смещений, `p.numel()` – число элементов тензора, условие `p.requires_grad` отбирает обучаемые (замороженные веса появятся в Б2.5). Результат сверяем с формулами из частей 1С и 3К (п. 3 теории).

Готовая ячейка ниже печатает параметры по слоям: `model.named_parameters()` дает пары «имя – тензор», например `features.0.weight` – веса модуля номер 0 внутри `features`.

In [ ]:
def count_params(model):
    return ...     # <-- сумма p.numel() по model.parameters() с p.requires_grad


mlp = MLP()
print(count_params(cnn), count_params(mlp), '  ожидается 9098 101770')
print(conv_params(1, 8, 3) + conv_params(8, 16, 3) + count_params_list([784, 10]), count_params_list([784, 128, 10]),
      '  ожидается 9098 101770 (формулы частей 1С и 3К)')
for name, p in cnn.named_parameters():
    print(f'{name:<22} {str(tuple(p.shape)):<16} {p.numel():>6}')

**Вопрос А2.** а) Какой слой `SimpleCNN` самый «тяжелый» и почему? б) Сколько параметров станет, если убрать оба слоя пулинга (свертки сохраняют размер $28 \times 28$)? Что это говорит о роли пулинга?

*Ваш ответ:*

**Задание А3. `train_epoch` и `evaluate` для `DataLoader`.**

Те же функции, что в части 2Г, но пакеты теперь дает `DataLoader` (п. 8 теории), поэтому перемешивать и нарезать вручную не нужно.
- `train_epoch(model, loader, loss_fn, optimizer, history=None)` – одна эпоха: `model.train()`, затем для каждого пакета `(x, y)` пять строк цикла обучения. Возвращает среднюю потерю по объектам: потерю пакета умножаем на его размер `len(y)`, складываем и делим на общее число объектов. Если передан список `history`, после каждого шага в него добавляется `loss.item()` – так в задании А6 мы нарисуем кривую потерь по шагам.
- `evaluate(model, loader, loss_fn)` – средняя потеря и доля верных ответов без обучения: `model.eval()` и `torch.no_grad()`. Число верных ответов в пакете: `(logits.argmax(dim=1) == y).sum().item()`.

*Проверка без обучения.* У необученной сети логиты почти одинаковы, вероятности всех 10 классов близки к $0{,}1$, поэтому потеря близка к $-\ln 0{,}1 = \ln 10 \approx 2{,}30$, а точность – к $0{,}1$ (угадывание).

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer, history=None):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        ...                                  # <-- обнулите градиенты
        loss = ...                           # <-- потеря на пакете
        ...                                  # <-- обратный проход
        ...                                  # <-- шаг оптимизатора
        total += loss.item() * len(y)
        n += len(y)
        if history is not None:
            history.append(loss.item())
    return total / n

def evaluate(model, loader, loss_fn):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            logits = ...                     # <-- выход модели
            total += loss_fn(logits, y).item() * len(y)
            correct += ...                   # <-- число верных ответов в пакете
            n += len(y)
    return total / n, correct / n


loss_fn = nn.CrossEntropyLoss()
loss0, acc0 = evaluate(cnn, test_loader, loss_fn)
print(round(loss0, 1), round(acc0, 1), '  ожидается около 2.3 и 0.1 (сеть пока угадывает)')

**Вопрос А3.** а) Почему потеря необученной сети близка к $\ln 10$? б) Зачем для оценки на обучающей выборке нужен отдельный `train_eval_loader` без перемешивания и с пакетом 256?

*Ваш ответ:*

**Задание А4. MLP против CNN.**

Функция `fit(model, epochs, lr)` обучает модель и возвращает строку таблицы – словарь с числом параметров, точностью на обучающей и тестовой выборке и временем обучения:
1. создать оптимизатор `torch.optim.Adam(model.parameters(), lr=lr)` и потерю `nn.CrossEntropyLoss()`;
2. `epochs` раз вызвать свою `train_epoch` и напечатать среднюю потерю эпохи;
3. посчитать точность своей `evaluate` на `train_eval_loader` и `test_loader`.

Готовая ячейка обучает обе сети по 2 эпохи (1–3 минуты на CPU) и печатает таблицу.

*Чего ожидать.* CNN с в 11 раз меньшим числом параметров должна дать более высокую точность на тесте – в этом проявляются заложенные в нее знания о картинках (п. 3 теории). Разница между точностью на обучении и на тесте показывает переобучение.

In [ ]:
def fit(model, epochs=2, lr=1e-3):
    optimizer = ...                           # <-- Adam по параметрам модели
    loss_fn = nn.CrossEntropyLoss()
    start = time.perf_counter()
    for epoch in range(epochs):
        loss = ...                            # <-- одна эпоха своей train_epoch
        print(f'  эпоха {epoch + 1}: потеря {loss:.4f}')
    seconds = time.perf_counter() - start
    _, acc_train = ...                        # <-- evaluate на train_eval_loader
    _, acc_test = ...                         # <-- evaluate на test_loader
    return {'параметры': count_params(model), 'train': acc_train, 'test': acc_test, 'время, с': seconds}

In [ ]:
# Готовая ячейка обучения: обе сети по 2 эпохи
results = {}
for name, make in [('MLP', MLP), ('CNN', SimpleCNN)]:
    torch.manual_seed(RANDOM_STATE)
    model = make()
    print(name)
    results[name] = fit(model, epochs=2)
    if name == 'CNN':
        cnn = model                           # обученная CNN нужна в задании А5

print(f'\n{"":<5}{"параметры":>10}{"train":>8}{"test":>8}{"время, с":>10}')
for name, r in results.items():
    print(f'{name:<5}{r["параметры"]:>10}{r["train"]:>8.3f}{r["test"]:>8.3f}{r["время, с"]:>10.1f}')
print('MLP test не ниже 0.83:', results['MLP']['test'] >= 0.83, '  ожидается True')
print('CNN test не ниже 0.86:', results['CNN']['test'] >= 0.86, '  ожидается True')
print('CNN точнее MLP:', results['CNN']['test'] > results['MLP']['test'], '  ожидается True')

**Вопрос А4.** а) Во сколько раз CNN меньше MLP по параметрам и во сколько раз дольше обучается? Объясните через число умножений (п. 3 теории). б) У какой сети больше разрыв между точностью на обучении и на тесте? в) Сравните с моделями 9 класса: какую точность, по-вашему, дала бы логистическая регрессия на 784 пикселях?

*Ваш ответ:*

**Задание А5. Что видят слои обученной сети.**

Функция `get_feature_maps(model, x)` возвращает пару карт признаков для пакета `x`:
- после первой свертки и ReLU – срез `model.features[:2]` (модули 0 и 1), форма $(N, 8, 28, 28)$;
- после второй свертки и ReLU – `model.features[:5]` (модули 0–4), форма $(N, 16, 14, 14)$.

Срез `nn.Sequential` – это тоже `nn.Sequential` из выбранных модулей, его можно вызывать как функцию. Считаем внутри `torch.no_grad()`: веса не меняются.

Готовые ячейки рисуют 8 фильтров первого слоя (`model.features[0].weight`, форма $(8, 1, 3, 3)$) и карты признаков одной тестовой картинки. Сравните фильтры с ядрами из части 3К: среди них обычно есть детекторы вертикальных и горизонтальных границ.

In [ ]:
def get_feature_maps(model, x):
    with torch.no_grad():
        maps1 = ...          # <-- первые два модуля features
        maps2 = ...          # <-- первые пять модулей features
    return maps1, maps2


img0 = x_batch[:1]                       # первая тестовая картинка, форма (1, 1, 28, 28)
maps1, maps2 = get_feature_maps(cnn, img0)
print(tuple(maps1.shape), tuple(maps2.shape), '  ожидается (1, 8, 28, 28) (1, 16, 14, 14)')
print(tuple(cnn.features[0].weight.shape), '  ожидается (8, 1, 3, 3)')
print(maps1.min().item() >= 0, '  ожидается True (после ReLU нет отрицательных чисел)')

In [ ]:
# Готовая ячейка: фильтры первого слоя и карты признаков
w1 = cnn.features[0].weight.detach()[:, 0]
fig, axs = plt.subplots(3, 8, figsize=(12, 5.2))
for i in range(8):
    axs[0, i].imshow(w1[i], cmap='RdGy_r'); axs[0, i].set_title(f'фильтр {i}', fontsize=9)
    axs[1, i].imshow(maps1[0, i], cmap='gray')
    axs[2, i].imshow(maps2[0, i], cmap='gray')
for ax in axs.ravel():
    ax.axis('off')
axs[1, 0].set_title('слой 1', fontsize=9, loc='left'); axs[2, 0].set_title('слой 2 (первые 8 из 16)', fontsize=9, loc='left')
fig.suptitle(f'Картинка: {CLASSES[int(y_batch[0])]}')
fig.tight_layout(); fig.savefig('cnn_filters_maps.png', dpi=100); plt.close(fig)
display(Image('cnn_filters_maps.png'))

**Вопрос А5.** а) Найдите среди фильтров первого слоя детекторы границ. Какой перепад яркости ищет каждый (по знакам весов)? б) Чем карты второго слоя отличаются от карт первого по размеру и по содержанию?

*Ваш ответ:*

**Задание А6. Остаточный блок.**

Класс `ResidualBlock(c)` – блок ResNet из п. 6 теории для $c$ каналов:
- в `__init__` – две свертки `nn.Conv2d(c, c, 3, padding=1)`: `self.conv1` и `self.conv2` (с `padding=1` размер карты не меняется – иначе $F(x)$ и $x$ нельзя было бы сложить);
- в `forward(x)`: $F(x) = \mathrm{conv2}(\mathrm{ReLU}(\mathrm{conv1}(x)))$, выход $\mathrm{ReLU}(F(x) + x)$.

Обычный блок `PlainBlock` дан готовым – он такой же, но без `+ x`. Готовый класс `DeepNet(block, n_blocks)` собирает сеть: входная свертка 1 → 16 с пулингом, 8 блоков, усреднение по карте и `Linear(16, 10)`. Параметров у обеих сетей поровну: сложение параметров не добавляет.

*Проверка «ничего не делать».* Если обнулить веса и смещение `conv2`, то $F(x) = 0$ и блок для неотрицательного входа возвращает сам $x$ – в точности тождественное отображение.

*Эксперимент.* Обе сети из 8 блоков (16 сверток подряд) обучаются одну эпоху на 6000 картинках методом SGD с одинаковыми сидами. У обычной сети сигнал и градиент затухают на каждом слое, и она почти не учится: потеря остается около $\ln 10 \approx 2{,}30$. У остаточной сети вход проходит по «прямой дороге» $+ x$, и потеря падает.

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        return torch.relu(self.conv2(torch.relu(self.conv1(x))))

class ResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = ...                  # <-- свертка c -> c, ядро 3, padding=1
        self.conv2 = ...                  # <-- еще одна такая же

    def forward(self, x):
        out = ...                         # <-- ReLU(conv1(x))
        out = ...                         # <-- conv2
        return ...                        # <-- ReLU(out + x)

class DeepNet(nn.Module):
    def __init__(self, block, n_blocks=8, c=16):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, c, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.blocks = nn.Sequential(*[block(c) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))


block = ResidualBlock(16)
x_pos = torch.rand(2, 16, 14, 14)
print(tuple(block(x_pos).shape), '  ожидается (2, 16, 14, 14)')
with torch.no_grad():
    block.conv2.weight.zero_(); block.conv2.bias.zero_()
print(torch.equal(block(x_pos), x_pos), '  ожидается True (F(x) = 0 – блок ничего не меняет)')
print(count_params(DeepNet(PlainBlock)), count_params(DeepNet(ResidualBlock)), '  ожидается 37450 37450')

In [ ]:
# Готовая ячейка эксперимента: 8 обычных блоков против 8 остаточных (около минуты)
small_ds = Subset(train_ds, range(6000))
curves = {}
for name, blk in [('обычные блоки', PlainBlock), ('остаточные блоки', ResidualBlock)]:
    torch.manual_seed(RANDOM_STATE)
    net = DeepNet(blk)
    loader = DataLoader(small_ds, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(RANDOM_STATE))
    opt = torch.optim.SGD(net.parameters(), lr=0.02, momentum=0.9)
    hist = []
    train_epoch(net, loader, nn.CrossEntropyLoss(), opt, history=hist)
    curves[name] = hist
    print(f'{name}: средняя потеря последних 20 шагов {np.mean(hist[-20:]):.3f}')
print('  ожидается: у обычных блоков около 2.30, у остаточных заметно ниже')

fig, ax = plt.subplots(figsize=(7, 3.4))
for (name, h), c in zip(curves.items(), ['#6E6E78', '#8F8260']):
    ax.plot(np.convolve(h, np.ones(5) / 5, mode='valid'), color=c, lw=2, label=name)
ax.axhline(np.log(10), color='#9A9AA3', ls='--', lw=1, label='ln 10')
ax.set_xlabel('шаг'); ax.set_ylabel('потеря'); ax.legend(); ax.set_title('16 сверток подряд: с остаточными связями и без')
fig.tight_layout(); fig.savefig('resnet_vs_plain.png', dpi=120); plt.close(fig)
display(Image('resnet_vs_plain.png'))

**Вопрос А6.** а) Почему обычная сеть из 16 сверток почти не учится, хотя параметров у нее столько же? Объясните через цепное правило из части 2Г. б) Почему добавление остаточного блока не может сделать сеть хуже?

*Ваш ответ:*

### Задание повышенной сложности (к части 4А)

1. **Матрица ошибок.** Посчитайте для обученной CNN матрицу $10 \times 10$: строка – верный класс, столбец – предсказанный. Какие пары классов сеть путает чаще всего? Покажите по 5 перепутанных картинок и объясните ошибки.
2. **Без пулинга.** Уберите оба `MaxPool2d` из `SimpleCNN` и поправьте классификатор. Проверьте ответ вопроса А2 через `count_params`, обучите сеть 2 эпохи и сравните точность и время с исходной.
3. **Шире или глубже.** Постройте две CNN примерно по $20\,000$ параметров: с более широкими слоями (больше каналов) и с тремя сверточными блоками. Сравните точность на тесте после 2 эпох.
4. **Предвестие Б2.2.** Добавьте `nn.BatchNorm2d(c)` после каждой свертки в `PlainBlock` и повторите эксперимент А6. Учится ли теперь обычная глубокая сеть?

In [ ]:
# Место для задания повышенной сложности